In [1]:
import requests
import pandas as pd
import time
from tqdm import tqdm


/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [2]:
coords_df = pd.read_csv("/mnt/d/SEM_six_SGP/crops/rain_data/district_latlon.csv")

coords_df.head()


,district,latitude,longitude
0,Ahmadabad,22.727305,72.203344
1,Amreli,21.421106,71.263665
2,Anand,22.450218,72.791052
3,Aravalli,23.469161,73.333194
4,BanasKantha,24.281052,72.039338


In [3]:
print("Total districts:", len(coords_df))


Total districts: 33


In [4]:
def fetch_rainfall(lat, lon, start_date, end_date):
    
    url = "https://archive-api.open-meteo.com/v1/archive"
    
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": "precipitation_sum",
        "timezone": "Asia/Kolkata"
    }
    
    response = requests.get(url, params=params)
    
    if response.status_code != 200:
        print("Error:", response.status_code)
        return None
    
    data = response.json()
    
    df = pd.DataFrame({
        "date": data["daily"]["time"],
        "rainfall_mm": data["daily"]["precipitation_sum"]
    })
    
    df["date"] = pd.to_datetime(df["date"])
    
    return df


In [7]:
all_district_data = []

for index, row in tqdm(coords_df.iterrows(), total=len(coords_df)):
    
    district = row["district"]
    lat = row["latitude"]
    lon = row["longitude"]
    
    df = fetch_rainfall(
        lat,
        lon,
        "2003-01-01",
        "2026-01-30"
    )
    
    if df is not None:
        df["district"] = district
        all_district_data.append(df)
    
    time.sleep(1)  # small delay to avoid API overload


100%|█████████████████████████████████████████████████████████████████████████| 33/33 [08:39<00:00, 15.74s/it]


In [8]:
rainfall_daily = pd.concat(all_district_data, ignore_index=True)

print(rainfall_daily.shape)
rainfall_daily.head()


(278223, 3)


,date,rainfall_mm,district
0,2003-01-01,0.0,Ahmadabad
1,2003-01-02,0.0,Ahmadabad
2,2003-01-03,0.0,Ahmadabad
3,2003-01-04,0.0,Ahmadabad
4,2003-01-05,0.0,Ahmadabad


In [9]:
rainfall_daily["year"] = rainfall_daily["date"].dt.year
rainfall_daily["month"] = rainfall_daily["date"].dt.month


In [10]:
rainfall_monthly = (
    rainfall_daily
    .groupby(["district", "year", "month"])
    .agg(
        monthly_rain_sum=("rainfall_mm", "sum"),
        monthly_rain_mean=("rainfall_mm", "mean"),
        monthly_rain_std=("rainfall_mm", "std"),
        rainy_days=("rainfall_mm", lambda x: (x > 0).sum())
    )
    .reset_index()
)

rainfall_monthly.head()


,district,year,month,monthly_rain_sum,monthly_rain_mean,monthly_rain_std,rainy_days
0,Ahmadabad,2003,1,0.1,0.003226,0.017961,1
1,Ahmadabad,2003,2,10.9,0.389286,2.059906,1
2,Ahmadabad,2003,3,0.0,0.000000,0.000000,0
3,Ahmadabad,2003,4,0.4,0.013333,0.073030,1
4,Ahmadabad,2003,5,0.0,0.000000,0.000000,0


In [14]:
def get_season(month):
    if month in [6, 7, 8, 9]:
        return "Kharif"
    elif month in [10, 11, 12, 1]:
        return "Rabi"
    else:
        return "Summer"

rainfall_monthly["season"] = rainfall_monthly["month"].apply(get_season)


In [15]:
rainfall_seasonal = (
    rainfall_monthly
    .groupby(["district", "year", "season"])
    .agg(
        season_rain_sum=("monthly_rain_sum", "sum"),
        season_rain_mean=("monthly_rain_mean", "mean"),
        season_rain_std=("monthly_rain_std", "mean")
    )
    .reset_index()
)

rainfall_seasonal.head()


,district,year,season,season_rain_sum,season_rain_mean,season_rain_std
0,Ahmadabad,2003,Kharif,698.6,5.662527,11.357841
1,Ahmadabad,2003,Rabi,0.7,0.005645,0.027238
2,Ahmadabad,2003,Summer,11.3,0.100655,0.533234
3,Ahmadabad,2004,Kharif,902.8,7.319355,16.755343
4,Ahmadabad,2004,Rabi,23.3,0.187930,0.484653


In [16]:
rainfall_daily.to_csv("/mnt/d/SEM_six_SGP/Crop_Price_V2/rain/gujarat_rainfall_daily_2005_2024.csv", index=False)

rainfall_monthly.to_csv("/mnt/d/SEM_six_SGP/Crop_Price_V2/rain/gujarat_rainfall_monthly_2005_2024.csv", index=False)

rainfall_seasonal.to_csv("/mnt/d/SEM_six_SGP/Crop_Price_V2/rain/gujarat_rainfall_seasonal_2005_2024.csv", index=False)


In [ ]:
!pip install earthengine-api geemap


In [4]:
!earthengine set_project melodic-bolt-473007-t7

/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
E0000 00:00:1771572312.355131    1344 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771572312.371766    1344 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS 

In [1]:
import ee
import geemap
import pandas as pd



/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


In [1]:
import ee

ee.Authenticate()

ee.Initialize(project='melodic-bolt-473007-t7')

print("Initialized Successfully 🚀")

/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/home/prince/miniconda3/envs/tf-gpu/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Initialized Successfully 🚀


In [2]:
print(ee.Number(1).getInfo())

1


In [3]:
india = ee.FeatureCollection("FAO/GAUL/2015/level2")
print(india.size().getInfo())

38258


In [6]:
india = ee.FeatureCollection("FAO/GAUL/2015/level2")

gujarat = india.filter(
    ee.Filter.eq("ADM1_NAME", "Gujarat")
)

print("Number of districts:",
      gujarat.size().getInfo())

Number of districts: 25


In [7]:
ndvi_collection = (
    ee.ImageCollection("MODIS/061/MOD13Q1")
    .select("NDVI")
    .filterDate("2005-01-01", "2024-12-31")
)

In [15]:
def scale_ndvi(image):
    ndvi = image.select("NDVI").multiply(0.0001)
    return ndvi.copyProperties(image, image.propertyNames())

ndvi_collection = ndvi_collection.map(scale_ndvi)

In [19]:
def monthly_ndvi(year, month):
    
    start = ee.Date.fromYMD(year, month, 1)
    end = start.advance(1, 'month')
    
    monthly_collection = ndvi_collection.filterDate(start, end)
    
    size = monthly_collection.size()
    
    # If collection has images
    def compute():
        monthly_img = monthly_collection.mean()
        
        stats = monthly_img.reduceRegions(
            collection=gujarat,
            reducer=ee.Reducer.mean(),
            scale=250
        )
        
        return stats.map(lambda f: f.set({
            "year": year,
            "month": month
        }))
    
    # If no images → return empty
    return ee.FeatureCollection(
        ee.Algorithms.If(
            size.gt(0),
            compute(),
            ee.FeatureCollection([])
        )
    )

In [20]:
features = []

for year in range(2003, 2025):
    for month in range(1, 13):
        result = monthly_ndvi(year, month)
        features.append(result)

ndvi_fc = ee.FeatureCollection(features).flatten()

In [21]:
task = ee.batch.Export.table.toDrive(
    collection=ndvi_fc,
    description="Gujarat_Monthly_NDVI_2005_2024",
    fileFormat="CSV"
)

task.start()